In [1]:
import os

PROXY_URL = "http://192.168.192.1:7897"

os.environ["http_proxy"] = PROXY_URL
os.environ["https_proxy"] = PROXY_URL
os.environ["HTTP_PROXY"] = PROXY_URL
os.environ["HTTPS_PROXY"] = PROXY_URL

print("Notebook proxy:", os.environ["HTTPS_PROXY"])

Notebook proxy: http://192.168.192.1:7897


In [2]:
from urllib.request import Request, urlopen

config_url = (
    "https://huggingface.co/"
    "distilbert/distilbert-base-uncased-finetuned-sst-2-english/"
    "resolve/714eb0fa89d2f80546fda750413ed43d93601a13/config.json"
)

request = Request(config_url, method="HEAD")

with urlopen(request, timeout=20) as response:
    print("HTTP status:", response.status)
    print("Final URL:", response.url)

HTTP status: 200
Final URL: https://huggingface.co/api/resolve-cache/models/distilbert/distilbert-base-uncased-finetuned-sst-2-english/714eb0fa89d2f80546fda750413ed43d93601a13/config.json?%2Fdistilbert%2Fdistilbert-base-uncased-finetuned-sst-2-english%2Fresolve%2F714eb0fa89d2f80546fda750413ed43d93601a13%2Fconfig.json=&etag=%22b57fe5dfcb8ec3f9bab35ed427c3434e3c7dd1ba%22


# Week 02：DistilBERT CPU 推理

目标：固定模型版本，在 CPU 上完成英文情感分类，并保存输入、原始输出、运行时间与失败案例。

In [3]:
import platform

import torch
import transformers
import huggingface_hub

print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("huggingface_hub:", huggingface_hub.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Device:", "cpu")

Python: 3.10.12
PyTorch: 2.14.0+cpu
Transformers: 5.17.0
huggingface_hub: 1.32.0
CUDA available: False
Device: cpu


## 1. 固定模型和设备

In [4]:
MODEL_ID = "distilbert/distilbert-base-uncased-finetuned-sst-2-english"
REVISION = "714eb0fa89d2f80546fda750413ed43d93601a13"
DEVICE = -1  # Transformers pipeline 中 -1 表示使用 CPU

print("Model:", MODEL_ID)
print("Revision:", REVISION)
print("Device: CPU")

Model: distilbert/distilbert-base-uncased-finetuned-sst-2-english
Revision: 714eb0fa89d2f80546fda750413ed43d93601a13
Device: CPU


## 2. 加载模型

In [5]:
import time
from transformers import pipeline

start = time.perf_counter()

classifier = pipeline(
    task="text-classification",
    model=MODEL_ID,
    revision=REVISION,
    device=DEVICE,
)

load_seconds = time.perf_counter() - start
print(f"Model loaded in {load_seconds:.2f} seconds")

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Model loaded in 1.95 seconds


## 3. 三组正常输入

In [6]:
normal_samples = [
    {"id": "normal_01", "text": "The documentation is clear and easy to follow."},
    {"id": "normal_02", "text": "The program crashes every time I open the file."},
    {"id": "normal_03", "text": "The update is useful, but the setup process is confusing."},
]

records = []

for sample in normal_samples:
    start = time.perf_counter()
    output = classifier(sample["text"])
    elapsed = time.perf_counter() - start

    record = {
        "id": sample["id"],
        "kind": "normal",
        "input": sample["text"],
        "raw_output": output,
        "elapsed_seconds": elapsed,
        "device": "cpu",
    }
    records.append(record)
    print(record)

{'id': 'normal_01', 'kind': 'normal', 'input': 'The documentation is clear and easy to follow.', 'raw_output': [{'label': 'POSITIVE', 'score': 0.9996985197067261}], 'elapsed_seconds': 0.8675074889999905, 'device': 'cpu'}
{'id': 'normal_02', 'kind': 'normal', 'input': 'The program crashes every time I open the file.', 'raw_output': [{'label': 'NEGATIVE', 'score': 0.9997367262840271}], 'elapsed_seconds': 0.16233482199999116, 'device': 'cpu'}
{'id': 'normal_03', 'kind': 'normal', 'input': 'The update is useful, but the setup process is confusing.', 'raw_output': [{'label': 'NEGATIVE', 'score': 0.998594343662262}], 'elapsed_seconds': 0.05089655000000448, 'device': 'cpu'}


## 4. 限制案例与无效输入

In [7]:
sarcasm_text = "Great, another two-hour delay. Just what I needed."

start = time.perf_counter()
sarcasm_output = classifier(sarcasm_text)
elapsed = time.perf_counter() - start

sarcasm_record = {
    "id": "limitation_01",
    "kind": "sarcasm_probe",
    "input": sarcasm_text,
    "human_interpretation": "negative / sarcastic",
    "raw_output": sarcasm_output,
    "elapsed_seconds": elapsed,
    "device": "cpu",
}

records.append(sarcasm_record)
print(sarcasm_record)

{'id': 'limitation_01', 'kind': 'sarcasm_probe', 'input': 'Great, another two-hour delay. Just what I needed.', 'human_interpretation': 'negative / sarcastic', 'raw_output': [{'label': 'POSITIVE', 'score': 0.9925507307052612}], 'elapsed_seconds': 0.11418046099998946, 'device': 'cpu'}


In [8]:
invalid_input = 12345

try:
    start = time.perf_counter()
    invalid_output = classifier(invalid_input)
    invalid_record = {
        "id": "invalid_01",
        "kind": "invalid_input",
        "input": invalid_input,
        "raw_output": invalid_output,
        "elapsed_seconds": time.perf_counter() - start,
    }
except Exception as exc:
    invalid_record = {
        "id": "invalid_01",
        "kind": "invalid_input",
        "input": invalid_input,
        "error_type": type(exc).__name__,
        "error_message": str(exc),
    }

records.append(invalid_record)
print(invalid_record)

{'id': 'invalid_01', 'kind': 'invalid_input', 'input': 12345, 'error_type': 'ValueError', 'error_message': 'text input must be of type `str` (single example), `list[str]` (batch or single pretokenized example) or `list[list[str]]` (batch of pretokenized examples) or `list[tuple[list[str], list[str]]]` (batch of pretokenized sequence pairs).'}


## 5. 保存原始结果

In [9]:
import json
from pathlib import Path

cwd = Path.cwd()
week02_dir = cwd if cwd.name == "week02" else cwd / "week02"
output_path = week02_dir / "predictions.jsonl"

with output_path.open("w", encoding="utf-8") as file:
    for record in records:
        file.write(json.dumps(record, ensure_ascii=False) + "\n")

print("Saved:", output_path)
print("Records:", len(records))

Saved: /home/yaxiaodang/multimodal-ai-bootcamp/week02/predictions.jsonl
Records: 5
